# Training Dynamics

Train a small model live and observe how loss, gradients,
and activations evolve. Demonstrates learning rate and
dropout effects.

**Prerequisites:** `uv sync --extra dev --extra analysis`

In [ ]:
import mlx.core as mx

from lmxlab.data.batching import batch_iterator
from lmxlab.data.tokenizer import CharTokenizer
from lmxlab.models.base import LanguageModel
from lmxlab.models.gpt import gpt_tiny
from lmxlab.training.callbacks import MetricsLogger
from lmxlab.training.config import TrainConfig
from lmxlab.training.trainer import Trainer

## Setup: Data and Model

In [ ]:
TEXT = (
    "To be or not to be that is the question "
    "whether tis nobler in the mind to suffer "
) * 50

tok = CharTokenizer(TEXT)
tokens = mx.array(tok.encode(TEXT), dtype=mx.int32)

# 90/10 split
split = int(0.9 * len(tokens))
train_tokens, val_tokens = tokens[:split], tokens[split:]
print(f"Train: {len(train_tokens)} tokens, Val: {len(val_tokens)} tokens")

In [ ]:
from dataclasses import replace

cfg = gpt_tiny()
cfg = replace(cfg, vocab_size=tok.vocab_size)
model = LanguageModel(cfg)
mx.eval(model.parameters())
print(f"Model: {model.count_parameters():,} params")

## Train and Collect Metrics

In [ ]:
train_config = TrainConfig(
    learning_rate=1e-3,
    max_steps=200,
    batch_size=4,
    warmup_steps=10,
    compile_step=True,
)

logger = MetricsLogger(log_interval=50)
trainer = Trainer(model, train_config, callbacks=[logger])

def make_data():
    yield from batch_iterator(train_tokens, batch_size=4, seq_len=32, shuffle=True)

history = trainer.train(make_data())
print(f"Final loss: {history[-1]['loss']:.4f}")

## Plot Loss Curve

In [ ]:
try:
    from lmxlab.analysis.plotting import plot_loss_curves

    losses = [h["loss"] for h in history]
    fig = plot_loss_curves(losses)
    fig.show()
except ImportError:
    print("Install matplotlib: uv sync --extra analysis")

## Gradient Flow Analysis

Check gradient magnitudes across layers to detect
vanishing or exploding gradients.

In [ ]:
try:
    from lmxlab.analysis.plotting import plot_gradient_flow

    fig = plot_gradient_flow(model)
    fig.show()
except ImportError:
    print("Install matplotlib: uv sync --extra analysis")

## Activation Norms by Layer

Check how activation magnitudes change through the model.
Healthy training should show roughly stable norms.

In [ ]:
from lmxlab.analysis.activations import ActivationCapture

test_tokens = mx.zeros((1, 16), dtype=mx.int32)
with ActivationCapture(model) as cap:
    model(test_tokens)

norms = cap.layer_norms()
print("Activation L2 norms by layer:")
for key, val in sorted(norms.items()):
    print(f"  {key}: {val:.4f}")

In [ ]:
try:
    from lmxlab.analysis.plotting import plot_layer_norms

    fig = plot_layer_norms(cap.activations)
    fig.show()
except ImportError:
    print("Install matplotlib: uv sync --extra analysis")

## Effect of Learning Rate

Compare training curves at different learning rates.

In [ ]:
results = {}
for lr in [1e-4, 1e-3, 1e-2]:
    mx.random.seed(42)
    m = LanguageModel(cfg)
    mx.eval(m.parameters())
    tc = TrainConfig(learning_rate=lr, max_steps=100, batch_size=4, warmup_steps=5, compile_step=False)
    t = Trainer(m, tc)
    h = t.train(batch_iterator(train_tokens, batch_size=4, seq_len=32, shuffle=True))
    results[f"lr={lr}"] = [s["loss"] for s in h]
    print(f"LR={lr}: final loss = {h[-1]['loss']:.4f}")

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(8, 4))
    for label, losses in results.items():
        ax.plot(losses, label=label)
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss")
    ax.set_title("Learning Rate Comparison")
    ax.legend()
    fig.show()
except ImportError:
    print("Install matplotlib for plots")